## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "environment" / "apa-python.yml").is_file()
    )
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/Wildtype_mice/2026")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import os

import pandas as pd
import scanpy as sc
import multiscoresplot as msp

import matplotlib.pyplot as plt
import seaborn as sns

# Load data


In [ ]:
BASEPATH = f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling"

In [ ]:
adata_counts = sc.read_h5ad(
    os.path.join(BASEPATH, "UTR_count_table_genelevel_bw12_above10cts_apa_named.h5ad")
)

adata_counts

In [ ]:
adata = sc.read_h5ad(
    os.path.join(BASEPATH, "UTR_count_table_genelevel_bw12_above10cts_apa_named_percentages.h5ad")
)

adata

In [ ]:
adata_counts.obs["batch_number"] = pd.Categorical(
    adata_counts.obs["batch_number"].astype(int),
    categories=sorted(adata_counts.obs["batch_number"].astype(int).unique()),
    ordered=True,
)
adata.obs["batch_number"] = pd.Categorical(
    adata.obs["batch_number"].astype(int),
    categories=sorted(adata.obs["batch_number"].astype(int).unique()),
    ordered=True,
)

In [ ]:
lineage_celltypes = [
    "Astrocytes",
    "qNSC1",
    "qNSC2",
    "ROB_qNSC2",
    "aNSC1",
    "aNSC2",
    "TAP",
    "NB",
    "Neurons",
]

adata_counts_lineage = adata_counts[adata_counts.obs["celltype"].isin(lineage_celltypes)].copy()

In [ ]:
adata_counts_lineage.obs.celltype.value_counts()

In [ ]:
adata_counts_lineage = adata_counts_lineage[
    ~adata_counts_lineage.obs["dpt_pseudotime"].isna()
].copy()

adata_counts_lineage

In [ ]:
adata_counts_lineage.obs.celltype.value_counts()

In [ ]:
# adata_counts_lineage.write_h5ad(
#     os.path.join(
#         BASEPATH,
#         "UTR_count_table_genelevel_bw12_above10cts_apa_named_lineage.h5ad",
#     )
# )

# Investigate the data


In [ ]:
adata.obs.celltype.value_counts()

In [ ]:
adata.obs

In [ ]:
adata.var

In [ ]:
# Check adata.var grouping by "name"
print("=== Checking adata.var grouping by gene ENSEMBL ID ===\n")

issues = {
    "multiple_gene_names": [],
    "multiple_utr_names": [],
    "multiple_transcript_ids": [],
    "non_sequential_idx": [],
}

for ensembl_id, group in adata.var.groupby("name", observed=False):
    # Check unique gene_name per group
    unique_gene_names = group["gene_name"].unique()
    if len(unique_gene_names) > 1:
        issues["multiple_gene_names"].append((ensembl_id, unique_gene_names.tolist()))

    # Check unique utr_name per group
    unique_utr_names = group["utr_name"].unique()
    if len(unique_utr_names) > 1:
        issues["multiple_utr_names"].append((ensembl_id, unique_utr_names.tolist()))

    # Check unique transcript_id per group
    unique_transcript_ids = group["transcript_id"].unique()
    if len(unique_transcript_ids) > 1:
        issues["multiple_transcript_ids"].append((ensembl_id, unique_transcript_ids.tolist()))

    # Check sequential idx starting from 1
    # Index format: {gene_name}_{idx}-{utr_size}
    idxs = []
    for index_val in group.index:
        try:
            # Extract idx: part between last "_" and "-"
            after_last_underscore = index_val.rsplit("_", 1)[-1]
            idx = int(after_last_underscore.split("-")[0])
            idxs.append(idx)
        except (ValueError, IndexError):
            idxs.append(None)

    if None not in idxs:
        expected = list(range(1, len(idxs) + 1))
        if sorted(idxs) != expected:
            issues["non_sequential_idx"].append((ensembl_id, idxs))
    else:
        issues["non_sequential_idx"].append((ensembl_id, idxs))

# Report results
for check, found_issues in issues.items():
    if found_issues:
        print(f"❌ FAILED - {check}: {len(found_issues)} group(s) with issues")
        for item in found_issues[:5]:  # Show first 5 issues to avoid flooding output
            print(f"ENSEMBL ID = '{item[0]}': {item[1]}")
        if len(found_issues) > 5:
            print(f"   ... and {len(found_issues) - 5} more")
    else:
        print(f"✅ PASSED - {check}: all groups are correct")

print(f"\nTotal groups (unique 'name' values): {adata.var['name'].nunique()}")
print(f"Total rows in adata.var: {len(adata.var)}")

In [ ]:
adata.var[adata.var["name"] == "ENSMUSG00000117310"]

In [ ]:
utr_sums = adata_counts.X.sum(axis=0)
utr_sums = utr_sums.A1 if hasattr(utr_sums, "A1") else utr_sums.ravel()

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(utr_sums, bins=100, ax=ax, color="steelblue", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Sum of counts per UTR (across cells)", fontsize=14)
ax.set_ylabel("Frequency", fontsize=14)
ax.set_yscale("log")
ax.set_title("Histogram of adata_counts.X.sum(axis=0)", fontsize=16)
ax.tick_params(axis="both", labelsize=12)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
utr_sums = adata_counts.X.sum(axis=1)
utr_sums = utr_sums.A1 if hasattr(utr_sums, "A1") else utr_sums.ravel()

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(utr_sums, bins=100, ax=ax, color="steelblue", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Sum of counts per cell (across UTRs)", fontsize=14)
ax.set_ylabel("Frequency", fontsize=14)
ax.set_yscale("log")
ax.set_title("Histogram of adata_counts.X.sum(axis=1)", fontsize=16)
ax.tick_params(axis="both", labelsize=12)
sns.despine()
plt.tight_layout()
plt.show()

# Dimensionality reduction


## Peaks - Non-normalized


In [ ]:
# PCA
sc.tl.pca(adata, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50)

# Neighborhood graph
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

# UMAP
sc.tl.umap(adata)

In [ ]:
lineage_celltypes = [
    "Astrocytes",
    "qNSC1",
    "qNSC2",
    "aNSC1",
    "aNSC2",
    "TAP",
    "NB",
    "Neurons",
]
adata_lineage = adata[adata.obs["celltype"].isin(lineage_celltypes)].copy()

In [ ]:
adata

In [ ]:
# Plot PCA
sc.pl.pca(
    adata,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
# Plot UMAP
sc.pl.umap(
    adata,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
sc.pl.umap(
    adata_lineage,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
sc.pl.umap(
    adata_lineage,
    color=[
        "leiden",
        "name_old",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

## Peaks - Normalized


In [ ]:
# Copy adata_counts to avoid overwriting the original
adata_norm = adata_counts.copy()

# Store raw counts
adata_norm.raw = adata_norm

# Normalize to library size (counts per 10k) and log transform
sc.pp.normalize_total(adata_norm, target_sum=1e4)
sc.pp.log1p(adata_norm)

# Scale the data (zero mean, unit variance) - important for PCA
sc.pp.scale(adata_norm, max_value=10)

# PCA
sc.tl.pca(adata_norm, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata_norm, log=True, n_pcs=50)

# Neighborhood graph
sc.pp.neighbors(adata_norm, n_neighbors=10, n_pcs=40)

# UMAP
sc.tl.umap(adata_norm)

In [ ]:
# Plot PCA
sc.pl.pca(
    adata_norm,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
# Plot UMAP
sc.pl.umap(
    adata_norm,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)